<a href="https://colab.research.google.com/github/Rohan0603/Daemon/blob/master/fine_tuning/notebooks/colab_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Qwen2.5-3B-Instruct for Daemon Desktop Pet (SFT)

This notebook fine-tunes `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` using Supervised Fine-Tuning (SFT) with Unsloth on a free Colab T4 GPU.

**Dataset:** Alpaca-format JSONL with instruction/input/output columns

**Setup:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Connect**
3. Run all cells in order

**Based on:** [unsloth-buddy](https://github.com/TYH-labs/unsloth-buddy) skill

## Cell 1: Install Unsloth & Dependencies

In [1]:
%%capture
!pip install unsloth
# Restart runtime if prompted (Runtime → Restart runtime)

## Cell 2: Verify GPU & Imports

In [2]:
import torch, json
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

from unsloth import FastLanguageModel
import unsloth, trl, transformers, datasets

print(json.dumps({
    "gpu": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "unsloth": unsloth.__version__,
    "trl": trl.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "cuda": torch.version.cuda,
}))
print("GPU ready!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
{"gpu": "Tesla T4", "vram_gb": 15.6, "unsloth": "2026.9.2", "trl": "0.24.0", "transformers": "5.5.0", "datasets": "4.3.0", "cuda": "12.8"}
GPU ready!


## Cell 3: Load Model & Apply LoRA

Loads `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` in 4-bit QLoRA (~2GB VRAM).

In [3]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.9.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Model loaded. Trainable params: None


## Cell 4: Prepare Dataset

**Upload your dataset file** (batch_00000_alpaca_clean.jsonl), then run the cell below.

Expected format: Alpaca JSONL with `instruction`, `input`, `output` columns.

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Upload your Alpaca-format JSONL dataset
# Supports instruction/input/output columns (Daemon dataset format)
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import files
import json
from datasets import Dataset

# Upload your JSONL file
print('Upload batch_00000_alpaca_clean.jsonl:')
uploaded = files.upload()

# Load the uploaded file
filename = list(uploaded.keys())[0]
with open(filename, 'r') as f:
    data = [json.loads(line) for line in f.readlines()]

dataset = Dataset.from_list(data)

print(f'Dataset: {len(dataset)} samples')
print(f'Columns: {dataset.column_names}')
print(f'Example: {dataset[0]}')

Upload batch_00000_alpaca_clean.jsonl:


Saving batch_00000_alpaca_clean.jsonl to batch_00000_alpaca_clean.jsonl
Dataset: 244 samples
Columns: ['instruction', 'input', 'output']
Example: {'instruction': 'Mode: desktop_companion\nAPM: 180\nIdle: 0s\nWindow: desktop\nScreen: Terminal test output\nMemory: user_habits: Uses AI for 90% of tasks | user_current_project: Daemon desktop pet\nTrigger: autonomous', 'input': '', 'output': "Oh we're DEFINING things now? Bold of you to start typing before your mouse calms the hell down!"}


## Cell 5: Train with SFT

In [7]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    instructions = example["instruction"]
    input_texts = example.get("input", "")
    outputs = example["output"]
    if isinstance(instructions, str):
        instructions = [instructions]
        input_texts = [input_texts]
        outputs = [outputs]
    formatted = []
    for instruction, input_text, output in zip(instructions, input_texts, outputs):
        user_content = instruction + (f"\n{input_text}" if input_text else "")
        messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output},
        ]
        formatted.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return formatted

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_func,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch size = 8
        max_steps = 300,                   # Increase for real training (e.g., 300-500)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        warmup_steps = 10,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/244 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## Cell 6: Save LoRA Adapters

In [8]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Adapters saved to lora_model/")

# List saved files
import os
for f in os.listdir("lora_model"):
    size = os.path.getsize(os.path.join("lora_model", f))
    print(f"  {f}: {size / 1e6:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


Adapters saved to lora_model/
  README.md: 0.0 MB
  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 119.8 MB
  chat_template.jinja: 0.0 MB
  tokenizer.json: 11.4 MB
  tokenizer_config.json: 0.0 MB


## Cell 7: Test Inference

Test with a Daemon-style prompt to verify the model learned the personality.

In [9]:
from unsloth import FastLanguageModel

# Reload for inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# Test with a Daemon-style prompt
messages = [
    {"role": "user", "content": "Mode: desktop_companion\nAPM: 120\nIdle: 30s\nWindow: vscode\nScreen: VS Code editor with Python source\nBrowser: https://github.com/\nMemory: user_habits: Uses AI for 90% of tasks | user_current_project: Daemon desktop pet\nTrigger: autonomous"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Mode: desktop_companion
APM: 120
Idle: 30s
Window: vscode
Screen: VS Code editor with Python source
Browser: https://github.com/
Memory: user_habits: Uses AI for 90% of tasks | user_current_project: Daemon desktop pet
Trigger: autonomous
assistant
I understand you're setting up a mode for me as an AI desktop companion, specifically tailored to be a daemon desktop pet. Let's break down the settings and trigger:

### Mode: Desktop Companion
This mode is designed to be present on your desktop, offering assistance or entertainment through various functions.

### APM (Activity Prediction Model): 120
This indicates that I have an APM score of 120, which means I'm highly effective at predicting and engaging in user activities. This score helps me anticipate and respond to your needs in real-time.

### Idle Time: 30 seconds
When I'm not actively interacting with you, I'll go into an idle state after 30 seconds, co

## Cell 8 (Optional): Export to GGUF

Export to GGUF format for use with Ollama, LM Studio, or llama.cpp.

In [ ]:
# Uncomment to export:
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF exported!")

# Download the GGUF file:
from google.colab import files
import glob

for f in glob.glob("model_gguf/*.gguf"):
    files.download(f)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in model_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:56<00:00, 28.08s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

## Cell 9 (Optional): Push to Hugging Face Hub

In [ ]:
Uncomment and set your HF token:
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
model.push_to_hub("your-username/qwen2.5-3b-finetuned", token=HF_TOKEN)
tokenizer.push_to_hub("your-username/qwen2.5-3b-finetuned", token=HF_TOKEN)

## Download Adapters

Download the `lora_model/` folder from the Colab file browser (left panel → folder icon → right-click → Download).